In [24]:
import os
import time
import pandas as pd
from IPython.display import display
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import hashlib
from datetime import datetime
from pymongo.write_concern import WriteConcern

In [10]:
# 1. Configuración de conexión a MongoDB

MONGO_URI = os.getenv(
    "MONGO_URI",
    "mongodb://mongo-primario:27017,mongo-secundario-1:27017,mongo-secundario-2:27017/?replicaSet=rs0"
)

cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
cliente.admin.command("ping")

db = cliente["Política"]
coleccion = db["Discursos"]

print("Conexión OK")
print("Documentos en Discursos:", coleccion.count_documents({}))

Conexión OK
Documentos en Discursos: 679


In [11]:
!sudo docker ps

CONTAINER ID   IMAGE                           COMMAND                  CREATED        STATUS                   PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   18 hours ago   Up 5 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
c3e0c073468a   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 5 minutes             0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp   mongo-primario
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 5 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 5 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1


In [12]:
def mostrar_estado_replica_set():
    estado = cliente.admin.command("replSetGetStatus")
    
    df_estado = pd.DataFrame([
        {
            "nodo": m["name"],
            "estado": m["stateStr"],
            "health": m["health"]
        }
        for m in estado["members"]
    ])
    
    return df_estado

mostrar_estado_replica_set()

,nodo,estado,health
0,mongo-primario:27017,PRIMARY,1.0
1,mongo-secundario-1:27017,SECONDARY,1.0
2,mongo-secundario-2:27017,SECONDARY,1.0


In [29]:
# 2. Cargar el mismo modelo de NLP usado para poblar la BD
print("Cargando modelo de NLP (all-MiniLM-L6-v2)...")
modelo = SentenceTransformer('all-MiniLM-L6-v2')

def buscar_discursos(consulta_texto, top_k=5):
    # Generar el embedding de la consulta del usuario
    embedding_consulta = modelo.encode([consulta_texto])
    
    # Recuperar todos los documentos de la base de datos
    documentos = list(coleccion.find({}))
    if not documentos:
        print("La base de datos está vacía. Ejecuta el poblamiento primero.")
        return

    # Extraer los embeddings y prepararlos para el cálculo matemático
    embeddings_db = [doc["embedding"] for doc in documentos]
    
    # Calcular similitud coseno usando scikit-learn
    # Esto compara la consulta contra TODOS los documentos a la vez
    similitudes = cosine_similarity(embedding_consulta, embeddings_db)[0]
    
    # Asociar cada documento con su puntaje de similitud
    resultados = []
    for i, doc in enumerate(documentos):
        resultados.append({
            "id": doc["_id"],
            "texto": doc["texto"][:200] + "...", # Mostramos solo un extracto de 200 caracteres
            "similitud": similitudes[i]
        })
        
    # Ordenar de mayor a menor similitud y tomar el Top K
    resultados_ordenados = sorted(resultados, key=lambda x: x["similitud"], reverse=True)[:top_k]
    
    # Imprimir los resultados por consola
    print(f"\nResultados Top {top_k} para: '{consulta_texto}'")
    print("="*60)
    for i, res in enumerate(resultados_ordenados, 1):
        print(f"{i}. Similitud Coseno: {res['similitud']:.4f} | ID (SHA-256): {res['id']}")
        print(f"   Extracto: {res['texto']}\n")

Cargando modelo de NLP (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
# Primera consulta de ejemplo
# equidad en los derechos humanos
# igualdad de género
# educación pública y futuro de Chile
# etc
consulta = input("Ingresa tu consulta textual (o 'salir' para terminar): ")
buscar_discursos(consulta)

Ingresa tu consulta textual (o 'salir' para terminar):  equidad en los derechos humanos



Resultados Top 5 para: 'equidad en los derechos humanos'
1. Similitud Coseno: 0.6174 | ID (SHA-256): 4303dad79e46173c8cfc0b10ddf5a23d8989ec27966894c7286167a8c8db088f
   Extracto: Muy buenos días:

 

Esto es un acto simple, pero con muchos símbolos y muchos significados.

 

Hoy estamos celebrando los 70 años desde que la humanidad, a través de Naciones Unidas, hizo la Declara...

2. Similitud Coseno: 0.5698 | ID (SHA-256): 2de4b4f6ac8e7db12fc61233d79749fe6a38799225f0ae1418acad2e24982bdf
   Extracto: Muy buenas tardes:

 

Nuestro Gobierno condena categóricamente los atropellos a los derechos humanos, en cualquier tiempo, en cualquier lugar y en cualquier circunstancia. 

 

Condenamos tanto los a...

3. Similitud Coseno: 0.5677 | ID (SHA-256): 8d408b4c0da96fde03d8b55589c940565300e5a5d6b38add42be36959963c29c
   Extracto: Buenos días:

 

Quiero expresar mi aprecio y valoración al informe de la Alta Comisionada para los Derechos Humanos de Naciones Unidas, Michelle Bachelet.

 

Creo q

# Probando la disponibilidad

In [20]:
!sudo docker stop mongo-primario
time.sleep(10)

mongo-primario


In [21]:
!sudo docker ps
mostrar_estado_replica_set()

CONTAINER ID   IMAGE                           COMMAND                  CREATED        STATUS                    PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   18 hours ago   Up 10 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 10 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 10 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1


,nodo,estado,health
0,mongo-primario:27017,(not reachable/healthy),0.0
1,mongo-secundario-1:27017,PRIMARY,1.0
2,mongo-secundario-2:27017,SECONDARY,1.0


In [23]:
# Probando la Operación de lectura 
cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
db = cliente["Política"]
coleccion = db["Discursos"]

consulta = input("Ingresa tu consulta textual (o 'salir' para terminar): ")
buscar_discursos(consulta)

Ingresa tu consulta textual (o 'salir' para terminar):  equidad en los derechos humanos



Resultados Top 5 para: 'equidad en los derechos humanos'
1. Similitud Coseno: 0.6174 | ID (SHA-256): 4303dad79e46173c8cfc0b10ddf5a23d8989ec27966894c7286167a8c8db088f
   Extracto: Muy buenos días:

 

Esto es un acto simple, pero con muchos símbolos y muchos significados.

 

Hoy estamos celebrando los 70 años desde que la humanidad, a través de Naciones Unidas, hizo la Declara...

2. Similitud Coseno: 0.5698 | ID (SHA-256): 2de4b4f6ac8e7db12fc61233d79749fe6a38799225f0ae1418acad2e24982bdf
   Extracto: Muy buenas tardes:

 

Nuestro Gobierno condena categóricamente los atropellos a los derechos humanos, en cualquier tiempo, en cualquier lugar y en cualquier circunstancia. 

 

Condenamos tanto los a...

3. Similitud Coseno: 0.5677 | ID (SHA-256): 8d408b4c0da96fde03d8b55589c940565300e5a5d6b38add42be36959963c29c
   Extracto: Buenos días:

 

Quiero expresar mi aprecio y valoración al informe de la Alta Comisionada para los Derechos Humanos de Naciones Unidas, Michelle Bachelet.

 

Creo q

In [26]:
# Probando la Operación de escritura
# Usamos write concern majority para demostrar escritura replicada
db_majority = cliente.get_database(
    "Política",
    write_concern=WriteConcern(w="majority", wtimeout=10000)
)

coleccion_majority = db_majority["Discursos"]

texto_nuevo = """
Discurso de prueba de alta disponibilidad.

Este documento fue insertado mientras el nodo mongo-primario estaba detenido.
La escritura fue recibida por el nuevo nodo primario elegido automáticamente
por el Replica Set de MongoDB.
""".strip()

hash_sha256 = hashlib.sha256(texto_nuevo.encode("utf-8")).hexdigest()

embedding_nuevo = modelo.encode(texto_nuevo).tolist()

documento_nuevo = {
    "_id": hash_sha256,
    "texto": texto_nuevo,
    "embedding": embedding_nuevo,
    "metadata": {
        "tipo": "prueba_alta_disponibilidad_escritura",
        "fecha_insercion": datetime.now().isoformat(),
        "primario_original_caido": True
    }
}

resultado = coleccion_majority.replace_one(
    {"_id": hash_sha256},
    documento_nuevo,
    upsert=True
)

print("Documento insertado correctamente.")
print("ID SHA-256:", hash_sha256)
print("Upserted ID:", resultado.upserted_id)

Documento insertado correctamente.
ID SHA-256: 6a8f9fa7fa9fd4ff874b7f09d8f3f99336dfae2625f7e846a5be60637a4b430c
Upserted ID: 6a8f9fa7fa9fd4ff874b7f09d8f3f99336dfae2625f7e846a5be60637a4b430c


In [30]:
# Lectura del documento insertado
doc = coleccion.find_one({"_id": hash_sha256})

if doc:
    print("Documento encontrado.")
    print("ID:", doc["_id"])
    print("Texto:")
    print(doc["texto"])
else:
    print("No se encontró el documento.")

Documento encontrado.
ID: 6a8f9fa7fa9fd4ff874b7f09d8f3f99336dfae2625f7e846a5be60637a4b430c
Texto:
Discurso de prueba de alta disponibilidad.

Este documento fue insertado mientras el nodo mongo-primario estaba detenido.
La escritura fue recibida por el nuevo nodo primario elegido automáticamente
por el Replica Set de MongoDB.


In [32]:
df = buscar_discursos("alta disponibilidad escritura replica set", top_k=5)


Resultados Top 5 para: 'alta disponibilidad escritura replica set'
1. Similitud Coseno: 0.4902 | ID (SHA-256): 6a8f9fa7fa9fd4ff874b7f09d8f3f99336dfae2625f7e846a5be60637a4b430c
   Extracto: Discurso de prueba de alta disponibilidad.

Este documento fue insertado mientras el nodo mongo-primario estaba detenido.
La escritura fue recibida por el nuevo nodo primario elegido automáticamente
p...

2. Similitud Coseno: 0.2552 | ID (SHA-256): 4e017783e202b9cd693553e1eb88f46c2a59276a72d16d0e35f3c9390a9516e5
   Extracto: Muchas gracias.

 

Bueno, es verdad, vamos a organizar la próxima reunión de la COP25 en Santiago y probablemente es la última oportunidad que tenemos para cambiar el curso del mundo en este respecto...

3. Similitud Coseno: 0.2513 | ID (SHA-256): 7d0d2c57193f473db26be4e6c8c39ac0e85bc0f6feb39fc3b4aca5569564017b
   Extracto: Queridos compatriotas:

 

Hoy, 5 de abril, nos hemos reunido para recordar y conmemorar el Bicentenario de la Batalla de Maipú, que fue la batalla decisiva

# Restaurar el nodo caído

In [33]:
!sudo docker start mongo-primario
time.sleep(10)
!sudo docker ps
mostrar_estado_replica_set()

mongo-primario
CONTAINER ID   IMAGE                           COMMAND                  CREATED        STATUS                    PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   18 hours ago   Up 16 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
c3e0c073468a   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 10 seconds             0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp   mongo-primario
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 16 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   18 hours ago   Up 16 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1


,nodo,estado,health
0,mongo-primario:27017,SECONDARY,1.0
1,mongo-secundario-1:27017,PRIMARY,1.0
2,mongo-secundario-2:27017,SECONDARY,1.0


In [34]:
doc = coleccion.find_one({"_id": hash_sha256})

if doc:
    print("Documento sigue disponible después de reactivar mongo-primario.")
    print("ID:", doc["_id"])
    print("Metadata:", doc.get("metadata", {}))
else:
    print("Documento no encontrado.")

Documento sigue disponible después de reactivar mongo-primario.
ID: 6a8f9fa7fa9fd4ff874b7f09d8f3f99336dfae2625f7e846a5be60637a4b430c
Metadata: {'tipo': 'prueba_alta_disponibilidad_escritura', 'fecha_insercion': '2026-06-01T21:28:39.274767', 'primario_original_caido': True}
